In [1]:
from dotenv import load_dotenv
load_dotenv()

True

---

#### 문서 로드

In [2]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/KCI_FI003153549_p5.pdf")
documents = loader.load()

#### 문서 분할

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splitted_documents = text_splitter.split_documents(documents)

#### 임베딩 모델(캐싱)

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

underlying_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    store,
    namespace = underlying_embeddings.model
)

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch06\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


#### 임베딩 & FAISS(Facebook AI Similarity Search) 벡터스토어 생성 및 저장

##### Case1. In-memory

In [5]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(splitted_documents, cached_embedder)

##### Case2. 로컬 디스크 저장(기존 파일 삭제 후 저장)

In [6]:
vectorstore = FAISS.from_documents(splitted_documents, cached_embedder)

# 영구적인 파일(persistent file)**로 디스크에 저장
# 기존 폴더에 새로운 인덱스 파일을 덮어쓰기 때문에 중복된 파일이 생성되지 않음(항상 가장 마지막에 저장된 벡터스토어의 파일만 존재)
vectorstore.save_local("./faiss_index")

In [7]:
vectorstore

In [8]:
vectorstore = None

In [9]:
vectorstore

In [10]:
# 벡터스토어 재로딩
vectorstore = FAISS.load_local(
    "./faiss_index", # 저장된 FAISS 인덱스 폴더의 경로
    cached_embedder,
    allow_dangerous_deserialization=True, #  FAISS 인덱스 내 데이터 역직렬화(deserialization) 허용(신뢰할 수 있는 파일 일 경우)
)

In [11]:
vectorstore

##### Case3. 로컬 디스크 저장(기존 파일이 있을 경우 로드)

In [12]:
vectorstore = None

In [13]:
vectorstore

In [14]:
import os

FAISS_INDEX_PATH = "./faiss_index"

if os.path.exists(FAISS_INDEX_PATH):
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH,
        underlying_embeddings,
        allow_dangerous_deserialization=True,
    )
else:
    # FAISS 벡터스토어 생성 및 저장
    vectorstore = FAISS.from_documents(splitted_documents, embedding_model)
    vectorstore.save_local(FAISS_INDEX_PATH)

In [15]:
vectorstore

---

In [16]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
# query = "국내에서 LLM을 임상시험에 적용한 대표적인 기관과 그 적용 사례를 2가지 이상 말해보세요."
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

In [17]:
results = vectorstore.similarity_search(query, k=5) # 검색을 외부에서 미리 실행한 후 반환된 결과 사용

In [18]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    '''다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트:{context}

질문: {question}
'''
)

prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='다음 컨텍스트만 사용해 질문에 답하세요.\n컨텍스트:{context}\n\n질문: {question}\n')

In [19]:
# 질문 예시
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

In [22]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model(
    "google_genai:gemini-2.5-flash",
)

chain = prompt | llm | StrOutputParser()

In [23]:
response = chain.invoke({'context': results, 'question': query})

In [24]:
print(response)

본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수는 **11,954페이지**입니다.

문서 유형별 비율은 다음과 같습니다:
*   **규제 문서**: 30%
*   **교육 자료**: 20%
*   **프로토콜 및 보고서**: 25%
*   **의료기기 특화 문서**: 15%
*   **기타**: 10%


---

In [25]:
# 리트리버 생성
retriever = vectorstore.as_retriever()

In [26]:
retriever.invoke(query)

[Document(id='97a0cb49-1d4e-4e24-98b0-f56a1dbfe4fb', metadata={'source': '../data/KCI_FI003153549_p5.pdf', 'file_path': '../data/KCI_FI003153549_p5.pdf', 'page': 0, 'total_pages': 1, 'CreationDate': 'D:20250909104709', 'Creator': 'PDFium', 'Producer': 'PDFium'}, page_content='1.2 Validity of Collected Data\n본 연구에서는 의료기기 임상시험에 특화된 Private\n수집된 데이터셋은 의료기기 임상시험에 특화된\nLLM 접근 방법을 제안한다. 이 접근 방법은 도메인 특화\nPrivate LLM 구축을 위해 도메인 적합성과 다양성, 그리\n데이터셋 구축, LLM 모델 튜닝, 도메인 특화 프롬프트 적\n고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총\n용, 그리고 도메인 특화 기능 구현의 네 가지 핵심 단계로\n111,954페이지로 구성된 데이터는 의료기기 임상시험의\n구성된다. 각 단계는 의료기기 임상시험 분야의 특수성을\n규제, 프로토콜 설계, 데이터 관리 등 전반적인 지식을 포\n반영하여 상호 유기적으로 작동하며, Figure 1와 같이 이\n괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수\n를 통해 해당 분야에서 최적의 성능을 달성하도록 설계되\n있는 다양한 시나리오를 반영하도록 설계되었다.\n었다.'),
 Document(id='61aa3b8b-4257-40ef-b7e9-75faaa14c9fc', metadata={'source': '../data/KCI_FI003153549_p5.pdf', 'file_path': '../data/KCI_FI003153549_p5.pdf', 'page': 0, 'total_pages': 1, 'CreationDate': 'D:20250909104709', 'Creator

In [ ]:
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model(
    "google_genai:gemini-2.5-flash",
)

# retriever, RunnablePassthrough 객체 전달
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt 
    | llm
    | StrOutputParser()
)

In [28]:
response = chain.invoke(query) # query는 RunnablePassthrough()를 통과하여 question이라는 키의 값이 됨

In [29]:
print(response)

본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수는 **11,954 페이지**이며, 총 158개의 문서로 구성되어 있습니다.

문서 유형별 비율은 다음과 같습니다:
*   규제 문서: 30%
*   교육 자료: 20%
*   프로토콜 및 보고서: 25%
*   의료기기 특화 문서: 15%
*   기타: 10%
